# Taller 1 - Computación Paralela (CUDA)
## KNN sobre Forest Cover Type

**Objetivo:** clasificar 20.000 parcelas de consulta usando como referencia otras 20.000 parcelas ya clasificadas, con el algoritmo de los **k vecinos más cercanos (KNN)** y distancia Euclidiana al cuadrado.

**Contenido del notebook:**
1. Descarga del dataset (Forest Cover Type).
2. Preparación de datos en Python (muestreo + normalización z-score + archivos `.bin`).
3. Programa en CUDA/C (`knn.cu`): versión secuencial en CPU + kernel paralelo.
4. Compilación y ejecución de las 8 combinaciones de bloques/hilos para k = 3, 15, 60 y 100.

> **Requisito:** en Colab active la GPU con `Entorno de ejecución -> Cambiar tipo de entorno -> Acelerador por hardware: GPU`.


## 1) Verificar GPU y descargar el dataset

Usamos el **mirror de UCI** (`covtype.data.gz`), que es exactamente el mismo conjunto `uciml/forest-cover-type-dataset` de Kaggle, pero se descarga directo sin necesidad de token.

* 581.012 parcelas, 54 variables + 1 etiqueta (7 tipos de cobertura).
* El archivo es texto separado por comas, sin encabezado.


In [ ]:
# Verificar que hay GPU disponible
!nvidia-smi

# Descargar el dataset (mirror UCI: idéntico al de Kaggle)
!wget -q -O covtype.data.gz https://archive.ics.uci.edu/ml/machine-learning-databases/covtype/covtype.data.gz
!gunzip -f covtype.data.gz
!ls -lh covtype.data


## 2) Preparación de los datos

Pasos:
1. Cargar el `covtype.data` (54 variables + etiqueta).
2. Tomar **20.000 parcelas de referencia** y **20.000 de consulta** con una permutación aleatoria (sin solape).
3. Normalizar con **z-score**: restar la media y dividir por la desviación estándar, calculadas **solo con la referencia**. Si alguna desviación es 0 se reemplaza por 1.
4. Exportar 3 archivos con `tofile()`: variables `float32` y etiquetas `int32`.


In [ ]:
import numpy as np
import pandas as pd
import os

# ----------------------------------------------------------------------
# 1) Cargar el dataset
#    covtype.data tiene 55 columnas separadas por coma:
#      54 variables (features) + 1 etiqueta (1..7)
# ----------------------------------------------------------------------
# pandas usa un parser en C: mucho mas rapido que np.loadtxt
datos = pd.read_csv("covtype.data", header=None, dtype=np.float64).to_numpy()
print("Forma del dataset completo:", datos.shape)

X = datos[:, :54].astype(np.float64)   # las 54 variables
y = datos[:, 54].astype(np.int32)      # etiqueta de cobertura (1..7)

# ----------------------------------------------------------------------
# 2) Muestreo aleatorio SIN solape: 20000 referencia + 20000 consulta
# ----------------------------------------------------------------------
rng = np.random.default_rng(42)
indices = rng.permutation(datos.shape[0])   # mezcla de todas las parcelas

idx_ref   = indices[:20000]
idx_query = indices[20000:40000]            # no se solapan con las de referencia

X_ref, y_ref = X[idx_ref], y[idx_ref]
X_query      = X[idx_query]
print("Referencia:", X_ref.shape, "| Consulta:", X_query.shape)

# ----------------------------------------------------------------------
# 3) Normalizacion z-score (media y desviacion SOLO de la referencia)
# ----------------------------------------------------------------------
media = X_ref.mean(axis=0)
desv  = X_ref.std(axis=0)
desv[desv == 0] = 1.0                       # evitar division por cero

X_ref_norm   = (X_ref   - media) / desv
X_query_norm = (X_query - media) / desv     # se usan los mismos estadisticos

# ----------------------------------------------------------------------
# 4) Exportar los 3 archivos binarios con tofile()
#    float32 para variables, int32 para etiquetas (orden C, sin cabecera)
# ----------------------------------------------------------------------
X_ref_norm.astype(np.float32).tofile("ref_vars.bin")
y_ref.astype(np.int32).tofile("ref_labels.bin")
X_query_norm.astype(np.float32).tofile("query_vars.bin")

print("\nArchivos generados:")
for nombre in ["ref_vars.bin", "ref_labels.bin", "query_vars.bin"]:
    print(f"  {nombre:16s} {os.path.getsize(nombre)/1e6:8.2f} MB")

# Comprobaciones rapidas
print("\nRango de etiquetas en referencia:", y_ref.min(), "a", y_ref.max())
print("Conteo por clase (referencia):", np.bincount(y_ref, minlength=8)[1:])
print("Ejemplo media normalizada (primeras 3 variables):",
      np.round(X_ref_norm.mean(axis=0)[:3], 6))


## 3) Programa en CUDA / C (`knn.cu`)

Características del programa:

* **C puro** (`stdio.h`, `stdlib.h`, `time.h`, `cuda_runtime.h`): sin clases, templates ni STL.
* **`knn_secuencial()`**: versión secuencial en CPU, línea base de comparación.
* **`knn_kernel()`**: cada hilo clasifica un **rango de parcelas** `[inicio, fin)` de la consulta.
  * Imprime **bloque, hilo y rango** cuando hay pocos hilos (≤128) para verificar el reparto.
* Para conservar los k menores se usa un **arreglo simple** `mejor_dist[k]` / `mejor_etiqueta[k]` (sin heaps).
* **Optimizaciones:** la fila de consulta se copia a registros una sola vez por parcela, el indice de la "peor" de las k distancias se mantiene incremental (O(1) amortizado en vez de O(k) por pareja), las etiquetas se guardan como `unsigned char` y los bucles de features van con `#pragma unroll`.
* **Votación**: gana la etiqueta más repetida; en empate gana la etiqueta de menor valor.
* Se miden: **secuencial, transferencia ida (H2D), kernel, transferencia vuelta (D2H), total y speedup**.
* 8 configuraciones × k = {3, 15, 60, 100}.

### Reparto del trabajo

Si hay `T = bloques * hilos` hilos en total, al hilo `tid` le corresponden las parcelas:

```
inicio = tid       * 20000 / T
fin    = (tid + 1) * 20000 / T
```

Así el reparto queda balanceado para cualquier T (incluso si T > 20000, algunos hilos no reciben ninguna parcela).


In [ ]:
%%writefile knn.cu
/* =====================================================================
 *  knn.cu  --  Clasificador KNN para Forest Cover Type
 *  Taller 1 de Computacion Paralela (CUDA)  --  lenguaje C puro
 *
 *  Compilar:  nvcc -O3 knn.cu -o knn
 *  Ejecutar:  ./knn
 *
 *  Lee tres archivos binarios generados en Python:
 *     ref_vars.bin    -> 20000 x 54 variables de referencia (float32)
 *     ref_labels.bin  -> 20000 etiquetas de referencia    (int32)
 *     query_vars.bin  -> 20000 x 54 variables de consulta (float32)
 *
 *  Optimizaciones frente a la version base:
 *    - La fila de consulta se copia a un arreglo local (registros) UNA vez
 *      por parcela, en lugar de releerla en cada una de las 20000 refs.
 *    - El indice de la "peor" de las k distancias se mantiene de forma
 *      incremental: solo se recalcula cuando hay reemplazo (O(1) amortizado
 *      por pareja, en vez de O(k) en cada pareja).
 *    - Las etiquetas se guardan como unsigned char (k <= 100 < 256).
 *    - Punteros __restrict__ y bucles de features con #pragma unroll.
 * ===================================================================== */
#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <cuda_runtime.h>

#define N_FEATURES 54
#define N_REF      20000
#define N_QUERY    20000
#define N_CLASSES  7
#define MAX_K      100          /* debe ser >= al mayor valor de k */

#define REF_VARS_FILE   "ref_vars.bin"
#define REF_LABEL_FILE  "ref_labels.bin"
#define QUERY_VARS_FILE "query_vars.bin"

/* ---------------------------------------------------------------
 * Distancia Euclidiana al cuadrado entre la fila de consulta qv
 * y una fila de referencia r.  El bucle esta desenrollado para que
 * qv quede en registros y se emitan varias cargas en paralelo.
 * Se usa tanto desde la CPU (__host__) como desde la GPU (__device__).
 * --------------------------------------------------------------- */
static __host__ __device__ __forceinline__ float distancia2(const float * __restrict__ qv,
                                                            const float * __restrict__ r)
{
    float d = 0.0f;
    #pragma unroll
    for (int f = 0; f < N_FEATURES; f++) {
        float dif = qv[f] - r[f];
        d += dif * dif;
    }
    return d;
}

/* =================================================================
 *  VERSION SECUENCIAL (CPU)
 * ================================================================= */
void knn_secuencial(const float * __restrict__ ref, const int * __restrict__ ref_labels,
                    const float * __restrict__ query, int * __restrict__ resultado, int k)
{
    for (int q = 0; q < N_QUERY; q++) {
        /* Copiar la parcela de consulta a un arreglo local (se reutiliza
           en las 20000 comparaciones en lugar de releerla cada vez). */
        float qv[N_FEATURES];
        for (int f = 0; f < N_FEATURES; f++) qv[f] = query[q * N_FEATURES + f];

        float         mejor_dist[MAX_K];      /* las k distancias mas pequenas */
        unsigned char mejor_etiqueta[MAX_K];  /* etiqueta de cada una de esas k */

        /* Las primeras k parcelas de referencia son las candidatas iniciales */
        for (int i = 0; i < k; i++) {
            mejor_dist[i]     = distancia2(qv, &ref[i * N_FEATURES]);
            mejor_etiqueta[i] = (unsigned char)ref_labels[i];
        }

        /* Indice de la peor (mayor distancia) de las k candidatas */
        int peor = 0;
        for (int i = 1; i < k; i++)
            if (mejor_dist[i] > mejor_dist[peor]) peor = i;

        /* Revisar el resto de las parcelas de referencia */
        for (int j = k; j < N_REF; j++) {
            float d = distancia2(qv, &ref[j * N_FEATURES]);
            if (d < mejor_dist[peor]) {
                mejor_dist[peor]     = d;
                mejor_etiqueta[peor] = (unsigned char)ref_labels[j];
                /* Solo cuando hay reemplazo hay que buscar la nueva peor */
                peor = 0;
                for (int i = 1; i < k; i++)
                    if (mejor_dist[i] > mejor_dist[peor]) peor = i;
            }
        }

        /* Votacion: gana la etiqueta mas repetida; empate -> menor etiqueta */
        int votos[N_CLASSES + 1];
        for (int i = 0; i <= N_CLASSES; i++) votos[i] = 0;
        for (int i = 0; i < k; i++) votos[mejor_etiqueta[i]]++;

        int elegida = 1, max_votos = votos[1];
        for (int i = 2; i <= N_CLASSES; i++) {
            if (votos[i] > max_votos) {
                max_votos = votos[i];
                elegida   = i;
            }
        }
        resultado[q] = elegida;
    }
}

/* =================================================================
 *  KERNEL CUDA
 *  Cada hilo clasifica un rango [inicio, fin) de parcelas de consulta.
 * ================================================================= */
__global__ void knn_kernel(const float * __restrict__ ref, const int * __restrict__ ref_labels,
                           const float * __restrict__ query, int * __restrict__ resultado,
                           int k, int total_hilos, int n_consulta)
{
    /* Identificador global del hilo */
    int tid = blockIdx.x * blockDim.x + threadIdx.x;

    /* Reparto del trabajo: rango de parcelas que le toca a este hilo */
    int inicio = (int)(((long long)tid       * n_consulta) / total_hilos);
    int fin    = (int)(((long long)(tid + 1) * n_consulta) / total_hilos);

    /* Impresion de verificacion: solo cuando hay pocos hilos (legible) */
    if (total_hilos <= 128) {
        printf("Bloque %2d | Hilo %2d | tid %3d -> parcelas [%5d, %5d)  (%d parcelas)\n",
               (int)blockIdx.x, (int)threadIdx.x, tid, inicio, fin, fin - inicio);
    }

    for (int q = inicio; q < fin; q++) {
        /* Copiar la parcela de consulta a registros (una sola vez) */
        float qv[N_FEATURES];
        #pragma unroll
        for (int f = 0; f < N_FEATURES; f++) qv[f] = query[q * N_FEATURES + f];

        float         mejor_dist[MAX_K];
        unsigned char mejor_etiqueta[MAX_K];

        for (int i = 0; i < k; i++) {
            mejor_dist[i]     = distancia2(qv, &ref[i * N_FEATURES]);
            mejor_etiqueta[i] = (unsigned char)ref_labels[i];
        }

        int peor = 0;
        for (int i = 1; i < k; i++)
            if (mejor_dist[i] > mejor_dist[peor]) peor = i;

        for (int j = k; j < N_REF; j++) {
            float d = distancia2(qv, &ref[j * N_FEATURES]);
            if (d < mejor_dist[peor]) {
                mejor_dist[peor]     = d;
                mejor_etiqueta[peor] = (unsigned char)ref_labels[j];
                peor = 0;
                for (int i = 1; i < k; i++)
                    if (mejor_dist[i] > mejor_dist[peor]) peor = i;
            }
        }

        int votos[N_CLASSES + 1];
        #pragma unroll
        for (int i = 0; i <= N_CLASSES; i++) votos[i] = 0;
        for (int i = 0; i < k; i++) votos[mejor_etiqueta[i]]++;

        int elegida = 1, max_votos = votos[1];
        for (int i = 2; i <= N_CLASSES; i++) {
            if (votos[i] > max_votos) {
                max_votos = votos[i];
                elegida   = i;
            }
        }
        resultado[q] = elegida;
    }
}

/* =================================================================
 *  MAIN
 * ================================================================= */
int main(void)
{
    /* -------- Arreglos en el host -------- */
    float *h_ref     = (float *)malloc((size_t)N_REF   * N_FEATURES * sizeof(float));
    float *h_query   = (float *)malloc((size_t)N_QUERY * N_FEATURES * sizeof(float));
    int   *h_ref_lab = (int   *)malloc((size_t)N_REF   * sizeof(int));
    int   *h_out_cpu = (int   *)malloc((size_t)N_QUERY * sizeof(int));
    int   *h_out_gpu = (int   *)malloc((size_t)N_QUERY * sizeof(int));

    if (!h_ref || !h_query || !h_ref_lab || !h_out_cpu || !h_out_gpu) {
        printf("Error: no se pudo reservar memoria en el host.\n");
        return 1;
    }

    /* -------- Leer los archivos binarios -------- */
    FILE *fp;
    size_t leidos;

    fp = fopen(REF_VARS_FILE, "rb");
    if (!fp) { printf("No se pudo abrir %s\n", REF_VARS_FILE); return 1; }
    leidos = fread(h_ref, sizeof(float), (size_t)N_REF * N_FEATURES, fp);
    fclose(fp);
    if (leidos != (size_t)N_REF * N_FEATURES) { printf("Error leyendo %s\n", REF_VARS_FILE); return 1; }

    fp = fopen(REF_LABEL_FILE, "rb");
    if (!fp) { printf("No se pudo abrir %s\n", REF_LABEL_FILE); return 1; }
    leidos = fread(h_ref_lab, sizeof(int), (size_t)N_REF, fp);
    fclose(fp);
    if (leidos != (size_t)N_REF) { printf("Error leyendo %s\n", REF_LABEL_FILE); return 1; }

    fp = fopen(QUERY_VARS_FILE, "rb");
    if (!fp) { printf("No se pudo abrir %s\n", QUERY_VARS_FILE); return 1; }
    leidos = fread(h_query, sizeof(float), (size_t)N_QUERY * N_FEATURES, fp);
    fclose(fp);
    if (leidos != (size_t)N_QUERY * N_FEATURES) { printf("Error leyendo %s\n", QUERY_VARS_FILE); return 1; }

    printf("Datos cargados: %d referencia x %d variables, %d consulta x %d variables\n",
           N_REF, N_FEATURES, N_QUERY, N_FEATURES);
    printf("Clases posibles: 1..%d\n\n", N_CLASSES);

    /* -------- Memoria en el device -------- */
    float *d_ref, *d_query;
    int   *d_ref_lab, *d_out;

    cudaMalloc((void **)&d_ref,     (size_t)N_REF   * N_FEATURES * sizeof(float));
    cudaMalloc((void **)&d_query,   (size_t)N_QUERY * N_FEATURES * sizeof(float));
    cudaMalloc((void **)&d_ref_lab, (size_t)N_REF   * sizeof(int));
    cudaMalloc((void **)&d_out,     (size_t)N_QUERY * sizeof(int));

    cudaEvent_t ev0, ev1;
    cudaEventCreate(&ev0);
    cudaEventCreate(&ev1);

    /* Transferencia inicial de referencia y etiquetas (UNA sola vez,
       porque no cambian entre pruebas y no se incluye en el tiempo de ida) */
    float ms_inicial = 0.0f;
    cudaEventRecord(ev0, 0);
    cudaMemcpy(d_ref,     h_ref,     (size_t)N_REF * N_FEATURES * sizeof(float), cudaMemcpyHostToDevice);
    cudaMemcpy(d_ref_lab, h_ref_lab, (size_t)N_REF * sizeof(int),                 cudaMemcpyHostToDevice);
    cudaEventRecord(ev1, 0);
    cudaEventSynchronize(ev1);
    cudaEventElapsedTime(&ms_inicial, ev0, ev1);
    printf("Transferencia inicial referencia+etiquetas: %.4f s\n\n", ms_inicial / 1000.0f);

    /* -------- Valores de k y configuraciones de bloques/hilos -------- */
    int k_values[4]    = {3, 15, 60, 100};
    int n_k            = 4;

    int cfg_bloques[8] = {1,    1,   100,   40,   200,   100,   625, 1000};
    int cfg_hilos[8]   = {1,   32,    10,   32,   100,   200,    32,  256};
    int n_cfg          = 8;

    int ik, ic;
    for (ik = 0; ik < n_k; ik++) {
        int k = k_values[ik];

        /* -------- Version secuencial en CPU (se mide una vez por cada k) -------- */
        clock_t c0 = clock();
        knn_secuencial(h_ref, h_ref_lab, h_query, h_out_cpu, k);
        clock_t c1 = clock();
        double seg_cpu = (double)(c1 - c0) / CLOCKS_PER_SEC;

        printf("=====================================================================\n");
        printf(" k = %d   |  secuencial CPU = %.4f s\n", k, seg_cpu);
        printf("=====================================================================\n");
        printf(" %8s %6s %12s %9s %10s %9s %10s %9s %6s\n",
               "bloques", "hilos", "secuencial", "H2D", "kernel", "D2H",
               "total", "speedup", "verif");
        printf("-------------------------------------------------------------------------\n");

        for (ic = 0; ic < n_cfg; ic++) {
            int bloques = cfg_bloques[ic];
            int hilos   = cfg_hilos[ic];
            int total_h = bloques * hilos;

            /* Transferencia de ida: variables de consulta */
            float ms_h2d = 0.0f;
            cudaEventRecord(ev0, 0);
            cudaMemcpy(d_query, h_query,
                       (size_t)N_QUERY * N_FEATURES * sizeof(float),
                       cudaMemcpyHostToDevice);
            cudaEventRecord(ev1, 0);
            cudaEventSynchronize(ev1);
            cudaEventElapsedTime(&ms_h2d, ev0, ev1);

            /* Kernel */
            float ms_kernel = 0.0f;
            cudaEventRecord(ev0, 0);
            knn_kernel<<<bloques, hilos>>>(d_ref, d_ref_lab, d_query, d_out,
                                           k, total_h, N_QUERY);
            cudaEventRecord(ev1, 0);
            cudaEventSynchronize(ev1);
            cudaEventElapsedTime(&ms_kernel, ev0, ev1);

            /* Transferencia de vuelta: etiquetas resultantes */
            float ms_d2h = 0.0f;
            cudaEventRecord(ev0, 0);
            cudaMemcpy(h_out_gpu, d_out,
                       (size_t)N_QUERY * sizeof(int),
                       cudaMemcpyDeviceToHost);
            cudaEventRecord(ev1, 0);
            cudaEventSynchronize(ev1);
            cudaEventElapsedTime(&ms_d2h, ev0, ev1);

            float ms_total = ms_h2d + ms_kernel + ms_d2h;

            /* Verificacion: comparar el resultado de la GPU contra la CPU */
            int iguales = 1;
            for (int q = 0; q < N_QUERY; q++) {
                if (h_out_gpu[q] != h_out_cpu[q]) { iguales = 0; break; }
            }

            double speedup = (ms_total > 0.0f) ? (seg_cpu / (ms_total / 1000.0)) : 0.0;

            printf(" %8d %6d %12.4f %9.4f %10.4f %9.4f %10.4f %8.2fx %6s\n",
                   bloques, hilos, seg_cpu,
                   ms_h2d / 1000.0f, ms_kernel / 1000.0f, ms_d2h / 1000.0f,
                   ms_total / 1000.0f, speedup, iguales ? "OK" : "DIF");
        }
        printf("\n");
    }

    /* -------- Liberar memoria -------- */
    cudaEventDestroy(ev0);
    cudaEventDestroy(ev1);
    cudaFree(d_ref);
    cudaFree(d_query);
    cudaFree(d_ref_lab);
    cudaFree(d_out);
    free(h_ref);
    free(h_query);
    free(h_ref_lab);
    free(h_out_cpu);
    free(h_out_gpu);

    return 0;
}


## 4) Compilar

Compilamos con `nvcc`. Se usa `-O3` para que la versión secuencial en CPU tenga un baseline razonable.


In [ ]:
!nvcc -O3 knn.cu -o knn && echo "Compilacion exitosa"


## 5) Ejecutar

Se ejecuta una sola vez y el programa recorre internamente:
* **4 valores de k**: 3, 15, 60 y 100.
* **8 configuraciones de bloques × hilos**: 1×1, 1×32, 100×10, 40×32, 200×100, 100×200, 625×32, 1000×256.

Para cada configuración imprime: **secuencial, H2D, kernel, D2H, total y speedup**, y verifica que el resultado de la GPU coincida con el de la CPU (`OK`).

> La impresión de "Bloque / Hilo / rango de parcelas" aparece **solo** en las dos configuraciones más pequeñas (1×1 y 1×32), para que la salida sea legible.

> ⚠️ La ejecución tarda varios minutos: la configuración **1×1** es un kernel escalar (un solo hilo en la GPU) y es la más lenta. Es normal.


In [ ]:
!./knn


## 6) Conclusiones (para completar en el informe)

Use las tablas impresas arriba y responda, por ejemplo:

1. **Configuración óptima**: ¿qué combinación de bloques/hilos dio el menor tiempo de kernel y total? (normalmente las que tienen muchos hilos, cerca de 20.000 = una parcela por hilo).
2. **Costo de las transferencias**: compare H2D + D2H frente al tiempo del kernel. Como los datos son pequeños (~4 MB por conjunto), las transferencias suelen ser muy rápidas comparadas con el cómputo.
3. **Efecto de k**: en la versión secuencial, ¿cómo crece el tiempo al pasar de k=3 a k=100? ¿Y en la paralela?
4. **¿Se justifica el paralelismo?**: compare el mejor tiempo GPU (total) contra el secuencial y calcule el speedup.
5. **Kernel escalar (1×1 vs CPU)**: un solo hilo de GPU es más lento que la CPU; explique por qué (frecuencia, caché, memoria).

### Notas sobre la implementación
* No se saca la raíz cuadrada en la distancia (basta comparar distancias al cuadrado).
* El arreglo de "k menores" se reemplaza por el peor candidato en cada paso (enfoque simple, sin heaps).
* Los empates en la votación se resuelven tomando la etiqueta de **menor valor** (se recorre de 1 a 7 con comparación estricta `>`).
* La referencia y las etiquetas se copian al device **una sola vez**; el tiempo de ida que se reporta por prueba corresponde a las variables de consulta.
